TODO: natively vectorize as many functions as possible to avoid iterations and speed up computation.

In [47]:
from engine import WordleEngine
import numpy as np
from constants import Square
from wordfreq import zipf_frequency
from scipy.stats import entropy


WORD_LEN = 5
LOCALE = 'en'
FREQ_VEC = np.vectorize(lambda word: zipf_frequency(word, LOCALE), otypes=[float])

engine = WordleEngine('words.txt', WORD_LEN)
df = engine._df
patterns = engine._patterns
df

,0,1,2,3,4
aahed,97,97,104,101,100
aalii,97,97,108,105,105
aapas,97,97,112,97,115
aargh,97,97,114,103,104
aarti,97,97,114,116,105
...,...,...,...,...,...
zuzim,122,117,122,105,109
zygal,122,121,103,97,108
zygon,122,121,103,111,110
zymes,122,121,109,101,115


Play a game of Worlde below. Record your guesses and responses as you make them and rerun the following cells to refresh the suggestions.

In [48]:
def convert_str_to_squares(abbrev: str) -> np.ndarray:
    """Convenience wrapper to save myself some copy pasting."""
    squares = []
    for letter in abbrev:
        assert letter in ('g', 'y', 'b')
        if letter == 'g':
            squares.append(Square.GREEN.value)
        elif letter == 'y':
            squares.append(Square.YELLOW.value)
        else:
            squares.append(Square.BLACK.value)
    return np.array(squares, dtype=np.uint8)

In [ ]:
for idx in range(patterns.shape[0])

In [ ]:
import pandas as pd
from cache import is_match

def filter_results(current: pd.DataFrame, guess: str, response: str) -> np.ndarray:
    squares = convert_str_to_squares(response)
    gs_np = df.loc[guess].to_numpy()
    mask = [is_match(gs_np, row.to_numpy(), squares) \
            for _, row in current.iterrows()]
    return np.array(mask)

In [ ]:
from scipy.stats import entropy
patterns = engine._patterns

# TODO: should be able to get actual entropy from guess.

def get_entropy(current: np.ndarray):
    entrops = np.empty((current.shape[1],), dtype=float)
    print(current.shape)
    for idx in range(current.shape[1]):
        _, counts = np.unique(current[idx, :, :], axis=0, return_counts=True)
        entrops[idx] = entropy(counts / current.shape[1], base=2)
    return entrops

In [55]:
guesses = [
    'tares',
    'sture',
]
responses = [
    'ybyyy',
    'ggbgg',
]

Iteratively filter the full list of words based on guesses and their square responses until we arrive at a subset of candidates. We can sort our suggestions to give the most likely candidates by using word frequency.

In [56]:
filtered = df.copy()
patts = np.copy(patterns)

for guess, response in zip(guesses, responses):
    mask = filter_results(filtered, guess, response)
    filtered = filtered[mask]
    patts = patts[:, mask, :]

recc = filtered.copy()
recc['freq'] = FREQ_VEC(recc.index)
recc.sort_values(by='freq', ascending=False, inplace=True)
print('Possible solutions sorted by log freq')
print(recc[:10], '=' * 35, sep='\n')

entrops = get_entropy(patts)
suggest = filtered.copy()
suggest['entropy'] = entrops
print('Most likely informative next guesses sorted by entropy')
suggest.sort_values('entropy', ascending=False, inplace=True)
print(suggest[:10], '=' * 35, sep='\n')

Possible solutions sorted by log freq
         0    1    2    3    4  freq
store  115  116  111  114  101  5.02
stere  115  116  101  114  101  1.20
stire  115  116  105  114  101  0.00
styre  115  116  121  114  101  0.00
(14855, 4, 5)


ValueError: Length of values (14855) does not match length of index (4)

Once you have the solution, visually ascertain that the produced squares match the game and validate our own function.

In [14]:
expected = 'right'

for guess, response in zip(guesses, responses):
    squares = engine.lookup_pattern(guess, expected)
    print(guess, ''.join(squares))

tares 🟨⬛🟨⬛⬛
about ⬛⬛⬛⬛🟩
grift 🟨🟨🟨⬛🟩
